# 神经网络的评估、优化

In [4]:
import torch
from torch import nn
import torchvision
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

## 1. 损失函数、反向传播

- L1损失函数

In [5]:
input = torch.tensor([1, 2, 3], dtype=torch.float32) # 浮点数类型才可以处理
target = torch.tensor([1, 2, 5], dtype=torch.float32)
input = torch.reshape(input, (1, 1, 1, 3))
target = torch.reshape(target, (1, 1, 1, 3)) # 注意形状，参考官方文档

In [6]:
loss_l1 = nn.L1Loss() # 默认参数reduction='mean'，均值L1损失函数
result = loss_l1(input, target)
result

tensor(0.6667)

- 均方差损失函数（MSEloss）

In [7]:
loss_mse = nn.MSELoss()
result = loss_mse(input, target)
result

tensor(1.3333)

- 交叉熵损失函数

In [8]:
x = torch.Tensor([0.1, 0.2, 0.3])
y = torch.Tensor([1])
x = torch.reshape(x, (1, 3)) # 注意输入形状

In [9]:
loss_cross = nn.CrossEntropyLoss()
result = loss_cross(x, y.long()) # 注意标签y的类型为long
result

tensor(1.1019)

In [10]:
class model_seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten(),
            nn.Linear(in_features=1024, out_features=64),
            nn.Linear(in_features=64, out_features=10),
        )

    def forward(self, x): # 用Sequential简洁很多，不容易出错
        x = self.seq(x)
        return x

- 与神经网络结合 + 反向传播

In [11]:
dataset = torchvision.datasets.CIFAR10(root='./my_cifar10', train=False, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [12]:
my_model_seq = model_seq()

In [13]:
for data in dataloader:
    imgs, targets = data
    output = my_model_seq(imgs)
    result_loss = loss_cross(output, targets)
    print(result_loss)
    result_loss.backward() # 反向传播，自动计算梯度并储存在result_loss里面
    print("ok")
    break

tensor(2.3066, grad_fn=<NllLossBackward0>)
ok


## 2. 优化器

- 以随机梯度下降（SGD）为例

In [ ]:
optimizer = torch.optim.SGD(my_model_seq.parameters(), lr=0.01) # 注意传入参数与学习率，注意.parameters()记得加括号

In [ ]:
for epoch in range(5): # 五轮优化
    running_loss = 0.0

    for data in dataloader:
        imgs, targets = data
        output = my_model_seq(imgs)
        result_loss = loss_cross(output, targets)

        optimizer.zero_grad() # 对上一步梯度清零，后面重新计算梯度
        result_loss.backward() # 反向传播
        optimizer.step() # 优化参数
        running_loss = running_loss + result_loss

    print(running_loss)

tensor(324.2296, grad_fn=<AddBackward0>)
tensor(315.7512, grad_fn=<AddBackward0>)
tensor(308.7707, grad_fn=<AddBackward0>)
tensor(302.3949, grad_fn=<AddBackward0>)
tensor(294.3714, grad_fn=<AddBackward0>)


*可以看到损失函数逐渐减小*